<a href="https://colab.research.google.com/github/Shameen-ghyas/Machine-Learning/blob/main/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instructions to Run This Notebook

### Before Running, One Time Setup

**Step 1:  Generate HuggingFace Token:**
- Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
- Click "New Token", Select "Fine-grained"
- Under Repositories, enable "Read access" and "Write access"
- Click "Create" and copy the token

**Step 2: Add Token to Colab Secrets:**
- Open the notebook in Google Colab
- Click the 🔑 icon in the left sidebar
- Click "Add new secret"
- Name: `HF_TOKEN`, Value: paste your token
- Toggle "Notebook access" to ON

**Step 3: Enable GPU:**
- Go to Runtime: Change runtime type
- Select "T4 GPU" under Hardware accelerator
- Click Save



### Running the Notebook

- Run all cells in order from top to bottom
- Cells 1–18 (data preparation + BERT evaluation) take approximately 5–10 minutes
- Phi-3 model download takes approximately 5 minutes
- Phi-3 inference takes approximately 10–15 minutes
- Do not close the browser tab while cells are running


### Notes
- GPU is required to run Phi-3, CPU will be too slow
- Dataset and trained BERT model are loaded automatically from HuggingFace, no manual file upload needed
- If runtime disconnects, simply re-run all cells from the top

Install

In [1]:
!pip install transformers datasets seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Imports

In [2]:
import json
import numpy as np
from collections import Counter
import random
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoModelForTokenClassification, TrainingArguments, AutoTokenizer, Trainer, DataCollatorForTokenClassification, pipeline, AutoModelForCausalLM
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.metrics import confusion_matrix
from huggingface_hub import HfApi,  hf_hub_download
from google.colab import userdata
import torch
import re

Data load from HuggingFace

In [3]:
# Download data files from HuggingFace
data_path = hf_hub_download(
    repo_id="Shameen-ghyas/pii-ner-dataset",
    filename="data.json",
    repo_type="dataset"
)

test_path = hf_hub_download(
    repo_id="Shameen-ghyas/pii-ner-dataset",
    filename="test_data.json",
    repo_type="dataset"
)

with open(data_path, 'r') as f:
    train_data = json.load(f)

with open(test_path, 'r') as f:
    test_data = json.load(f)

print("Data loaded from HuggingFace!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


data.json:   0%|          | 0.00/34.0M [00:00<?, ?B/s]

test_data.json: 0.00B [00:00, ?B/s]

Data loaded from HuggingFace!


Dataset Inspect

In [4]:
# total number of samples
print("Total samples:", len(train_data))

# Preview the first entry
print("\nFirst entry:")
print("Tokens:", train_data[0]['tokens'])
print("Tags:  ", train_data[0]['ner_tags'])
print("Sentence:", train_data[0]['sequence'])

Total samples: 28516

First entry:
Tokens: ['Since', 'then', ',', 'only', 'Terry', 'Bradshaw', 'in', '147', 'games', ',', 'Joe', 'Montana', 'in', '139', 'games', ',', 'and', 'Tom', 'Brady', 'in', '131', 'games', 'have', 'reached', '100', 'wins', 'more', 'quickly', '.']
Tags:   ['O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Sentence: Since then , only Terry Bradshaw in 147 games , Joe Montana in 139 games , and Tom Brady in 131 games have reached 100 wins more quickly .


Tag Distribution

In [5]:
# Collect all NER tags from the entire dataset
all_tags = []
for entry in train_data:
    all_tags.extend(entry['ner_tags'])

# Count each tag
tag_counts = Counter(all_tags)
print("Tag distribution in training data:")
for tag, count in tag_counts.items():
    print(f"  {tag}: {count}")

# Specifically check if any email tags exist
email_present = any('EMAIL' in tag for tag in all_tags)
print("\nEmail tags present in dataset?", email_present)

Tag distribution in training data:
  O: 639055
  B-PER: 40264
  I-PER: 29466

Email tags present in dataset? False


Email Functions

In [6]:
# Common email domains for realistic email generation
domains = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'mail.com']

def name_to_email(name_tokens):
    # Convert name tokens to a realistic email address
    # e.g. ["Brad", "Wilk"] => "brad.wilk@gmail.com"
    name_part = '.'.join([n.lower() for n in name_tokens])
    domain = random.choice(domains)
    return f"{name_part}@{domain}"

def extract_names(entry):
    # Extract all person name spans from a single dataset entry
    names = []
    current_name = []
    for token, tag in zip(entry['tokens'], entry['ner_tags']):
        if tag == 'B-PER':
            if current_name:
                names.append(current_name)
            current_name = [token]
        elif tag == 'I-PER':
            current_name.append(token)
        else:
            if current_name:
                names.append(current_name)
                current_name = []
    if current_name:
        names.append(current_name)
    return names

# Test it on the first entry
names_found = extract_names(train_data[0])
print("Names found in first entry:", names_found)

# Generate emails from those names
for name in names_found:
    print(f"  {name} → {name_to_email(name)}")

Names found in first entry: [['Terry', 'Bradshaw'], ['Joe', 'Montana'], ['Tom', 'Brady']]
  ['Terry', 'Bradshaw'] → terry.bradshaw@gmail.com
  ['Joe', 'Montana'] → joe.montana@yahoo.com
  ['Tom', 'Brady'] → tom.brady@outlook.com


Inject Email Function

In [7]:
def inject_email_into_entry(entry):
    # Extract names from this entry
    names = extract_names(entry)

    # If no names found, skip this entry
    if not names:
        return None

    # Pick one random name and generate an email for it
    chosen_name = random.choice(names)
    email = name_to_email(chosen_name)

    email_tokens = email.replace('@', ' @ ').replace('.', ' . ').split()

    # Create email tags: first token is B-EMAIL, rest are I-EMAIL
    email_tags = ['B-EMAIL'] + ['I-EMAIL'] * (len(email_tokens) - 1)

    # Add a natural phrase before the email + inject email tokens at the end
    prefix = ['Contact', 'at']
    prefix_tags = ['O', 'O']

    new_tokens = entry['tokens'] + prefix + email_tokens
    new_tags = entry['ner_tags'] + prefix_tags + email_tags
    new_sequence = entry['sequence'] + f" Contact at {email}"

    return {
        'lang': entry['lang'],
        'tokens': new_tokens,
        'ner_tags': new_tags,
        'sequence': new_sequence
    }

# Test on first entry
result = inject_email_into_entry(train_data[0])
print("New tokens:", result['tokens'])
print("New tags:  ", result['ner_tags'])
print("New sequence:", result['sequence'])

New tokens: ['Since', 'then', ',', 'only', 'Terry', 'Bradshaw', 'in', '147', 'games', ',', 'Joe', 'Montana', 'in', '139', 'games', ',', 'and', 'Tom', 'Brady', 'in', '131', 'games', 'have', 'reached', '100', 'wins', 'more', 'quickly', '.', 'Contact', 'at', 'terry', '.', 'bradshaw', '@', 'yahoo', '.', 'com']
New tags:   ['O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL', 'I-EMAIL']
New sequence: Since then , only Terry Bradshaw in 147 games , Joe Montana in 139 games , and Tom Brady in 131 games have reached 100 wins more quickly . Contact at terry.bradshaw@yahoo.com


Augment Data

In [8]:
# Apply email injection to 30% of training data randomly
# We don't want ALL entries to have emails, that would be unrealistic
# In real world, only some text contains email addresses

random.seed(42)

augmented_data = []

for entry in train_data:
    if random.random() < 0.3:
        new_entry = inject_email_into_entry(entry)
        if new_entry:
            augmented_data.append(new_entry)
        else:
            augmented_data.append(entry)
    else:
        augmented_data.append(entry)

# Count how many entries now have emails
email_count = sum(1 for e in augmented_data
                  if any('EMAIL' in t for t in e['ner_tags']))

print(f"Total entries: {len(augmented_data)}")
print(f"Entries with emails injected: {email_count}")
print(f"Entries without emails: {len(augmented_data) - email_count}")

Total entries: 28516
Entries with emails injected: 8412
Entries without emails: 20104


Train/Val Split


In [9]:
# Split augmented data into 80% train and 20% validation
train_split, val_split = train_test_split(
    augmented_data,
    test_size=0.2,
    random_state=42
)

print(f"Training samples:   {len(train_split)}")
print(f"Validation samples: {len(val_split)}")

Training samples:   22812
Validation samples: 5704


Tokenizer Load

In [10]:
# Loading pretrained BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 28996


Label Mappings


In [11]:
# All possible NER tags in our dataset
label_list = ['O', 'B-PER', 'I-PER', 'B-EMAIL', 'I-EMAIL']

# Convert tag names to numbers
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for idx, label in enumerate(label_list)}

print("Label to ID mapping:", label2id)
print("ID to Label mapping:", id2label)

Label to ID mapping: {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-EMAIL': 3, 'I-EMAIL': 4}
ID to Label mapping: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-EMAIL', 4: 'I-EMAIL'}


Tokenize Function

In [12]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        padding='max_length',
        max_length=128
    )

    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Ignore special tokens
            elif word_idx != previous_word_idx:
                # First token of a word — assign actual label
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)  # Ignore subsequent subword tokens
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs['labels'] = labels
    return tokenized_inputs

print("Tokenization function defined successfully!")

Tokenization function defined successfully!


HuggingFace Dataset Convert

In [13]:
# Convert our Python lists to HuggingFace Dataset format
train_dataset = Dataset.from_list(train_split)
val_dataset = Dataset.from_list(val_split)

print("Train dataset:", train_dataset)
print("Val dataset:", val_dataset)

Train dataset: Dataset({
    features: ['lang', 'ner_tags', 'sequence', 'tokens'],
    num_rows: 22812
})
Val dataset: Dataset({
    features: ['lang', 'tokens', 'ner_tags', 'sequence'],
    num_rows: 5704
})


Apply Tokenization

In [14]:
# Apply our tokenization function to entire train and val datasets
train_tokenized = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_dataset.column_names
)

val_tokenized = val_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=val_dataset.column_names
)

print("Train tokenized:", train_tokenized)
print("Val tokenized:", val_tokenized)

Map:   0%|          | 0/22812 [00:00<?, ? examples/s]

Map:   0%|          | 0/5704 [00:00<?, ? examples/s]

Train tokenized: Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 22812
})
Val tokenized: Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5704
})


Training Arguments

In [15]:
training_args = TrainingArguments(
    output_dir='./ner_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.01,
    seed=42
)

print("Training arguments set successfully!")

Training arguments set successfully!


Compute Metrics

In [16]:
def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    # Get the highest scoring label for each token
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for prediction, label in zip(predictions, labels):
        current_true = []
        current_pred = []
        for pred_id, label_id in zip(prediction, label):
            if label_id == -100:
                continue  # Skip special tokens
            current_true.append(id2label[label_id])
            current_pred.append(id2label[pred_id])
        true_labels.append(current_true)
        true_predictions.append(current_pred)

    return {
        'f1': f1_score(true_labels, true_predictions),
        'precision': precision_score(true_labels, true_predictions),
        'recall': recall_score(true_labels, true_predictions)
    }

print("Metrics function defined successfully!")

Metrics function defined successfully!


Load Model from HF

In [17]:
#load model from HuggingFace
model = AutoModelForTokenClassification.from_pretrained("Shameen-ghyas/pii-ner-bert")
tokenizer = AutoTokenizer.from_pretrained("Shameen-ghyas/pii-ner-bert")

config.json:   0%|          | 0.00/973 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/431M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Trainer

In [18]:
# NOTE: Model was trained separately and saved to HuggingFace
# Training code reference:
# trainer.train(): 3 epochs, bert-base-cased, lr=2e-5
# Results: F1=98.56%, Precision=98.38%, Recall=98.74%

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer recreated successfully!")

Trainer recreated successfully!


Validation Evaluation

In [19]:
# Get predictions on validation set
predictions, labels, _ = trainer.predict(val_tokenized)
predictions = np.argmax(predictions, axis=2)

true_labels = []
true_predictions = []

for prediction, label in zip(predictions, labels):
    current_true = []
    current_pred = []
    for pred_id, label_id in zip(prediction, label):
        if label_id == -100:
            continue
        current_true.append(id2label[label_id])
        current_pred.append(id2label[pred_id])
    true_labels.append(current_true)
    true_predictions.append(current_pred)

# Per-class F1, Precision, Recall
print("=" * 35)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 35)
print(classification_report(true_labels, true_predictions))

# FPR and FNR per class
print("=" * 35)
print("FPR AND FNR PER CLASS")
print("=" * 35)

# Flatten for token-level metrics
flat_true = [t for seq in true_labels for t in seq]
flat_pred = [p for seq in true_predictions for p in seq]

entity_labels = ['B-PER', 'I-PER', 'B-EMAIL', 'I-EMAIL']

for entity in entity_labels:
    TP = sum(1 for t, p in zip(flat_true, flat_pred) if t == entity and p == entity)
    FP = sum(1 for t, p in zip(flat_true, flat_pred) if t != entity and p == entity)
    FN = sum(1 for t, p in zip(flat_true, flat_pred) if t == entity and p != entity)
    TN = sum(1 for t, p in zip(flat_true, flat_pred) if t != entity and p != entity)

    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
    FNR = FN / (FN + TP) if (FN + TP) > 0 else 0

    print(f"\n{entity}:")
    print(f"  FPR (False Positive Rate): {FPR:.4f}")
    print(f"  FNR (False Negative Rate): {FNR:.4f}")

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

       EMAIL       1.00      1.00      1.00      1669
         PER       0.98      0.99      0.98      8072

   micro avg       0.98      0.99      0.98      9741
   macro avg       0.99      0.99      0.99      9741
weighted avg       0.98      0.99      0.98      9741

FPR AND FNR PER CLASS

B-PER:
  FPR (False Positive Rate): 0.0011
  FNR (False Negative Rate): 0.0090

I-PER:
  FPR (False Positive Rate): 0.0003
  FNR (False Negative Rate): 0.0072

B-EMAIL:
  FPR (False Positive Rate): 0.0000
  FNR (False Negative Rate): 0.0000

I-EMAIL:
  FPR (False Positive Rate): 0.0000
  FNR (False Negative Rate): 0.0000


Test Data Inspect

In [20]:
# Load the test dataset

print("Test samples:", len(test_data))
print("\nFirst entry:")
print("Tokens:", test_data[0]['tokens'])
print("Tags:  ", test_data[0]['ner_tags'])

Test samples: 3650

First entry:
Tokens: ['included', 'future', 'Rage', 'Against', 'the', 'Machine', 'and', 'Audioslave', 'drummer', 'Brad', 'Wilk', '.']
Tags:   ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O']


Test Set Evaluation

In [21]:
# Convert test data to HuggingFace Dataset format
test_dataset = Dataset.from_list(test_data)

# Apply same tokenization as training data
test_tokenized = test_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=test_dataset.column_names
)

# Get predictions on test set
test_predictions, test_labels, _ = trainer.predict(test_tokenized)
test_predictions = np.argmax(test_predictions, axis=2)

# Align predictions with true labels
test_true_labels = []
test_true_predictions = []

for prediction, label in zip(test_predictions, test_labels):
    current_true = []
    current_pred = []
    for pred_id, label_id in zip(prediction, label):
        if label_id == -100:
            continue
        current_true.append(id2label[label_id])
        current_pred.append(id2label[pred_id])
    test_true_labels.append(current_true)
    test_true_predictions.append(current_pred)

# Final evaluation report
print("=" * 50)
print("FINAL TEST SET EVALUATION — BERT")
print("=" * 50)
print(classification_report(test_true_labels, test_true_predictions))

# FPR and FNR on test set
print("=" * 50)
print("FPR AND FNR ON TEST SET")
print("=" * 50)

flat_true = [t for seq in test_true_labels for t in seq]
flat_pred = [p for seq in test_true_predictions for p in seq]

for entity in ['B-PER', 'I-PER']:
    TP = sum(1 for t, p in zip(flat_true, flat_pred) if t == entity and p == entity)
    FP = sum(1 for t, p in zip(flat_true, flat_pred) if t != entity and p == entity)
    FN = sum(1 for t, p in zip(flat_true, flat_pred) if t == entity and p != entity)
    TN = sum(1 for t, p in zip(flat_true, flat_pred) if t != entity and p != entity)

    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
    FNR = FN / (FN + TP) if (FN + TP) > 0 else 0

    print(f"\n{entity}:")
    print(f"  FPR: {FPR:.4f}")
    print(f"  FNR: {FNR:.4f}")

Map:   0%|          | 0/3650 [00:00<?, ? examples/s]

FINAL TEST SET EVALUATION — BERT
              precision    recall  f1-score   support

         PER       0.97      0.99      0.98      5210

   micro avg       0.97      0.99      0.98      5210
   macro avg       0.97      0.99      0.98      5210
weighted avg       0.97      0.99      0.98      5210

FPR AND FNR ON TEST SET

B-PER:
  FPR: 0.0015
  FNR: 0.0094

I-PER:
  FPR: 0.0003
  FNR: 0.0061


Zero-Shot LLM (Load PHI-3)

In [22]:
hf_token = userdata.get('HF_TOKEN')

# Load Phi-3 tokenizer AND model — both same model!
phi_tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    token=hf_token
)

phi_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    torch_dtype=torch.float16,  # Half precision to save GPU memory
    device_map="auto",
    token=hf_token
)

print("Phi-3 loaded successfully!")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Phi-3 loaded successfully!


Masking function

In [23]:
def mask_pii_with_llm(sentence):
    # Phi-3 uses its own chat format
    prompt = f"""<|system|>
You are a PII masking assistant. Replace person names with [NAME]. When an email address is present, replace the *entire* email address with [EMAIL] and *do not* mask any part of the email address as [NAME]. Output only the masked sentence, nothing else.<|end|>
<|user|>
{sentence}<|end|>
<|assistant|>
"""

    inputs = phi_tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = phi_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,        # Greedy decoding for consistency
            pad_token_id=phi_tokenizer.eos_token_id
        )

    # Decode only newly generated tokens
    input_length = inputs['input_ids'].shape[1]
    new_tokens = outputs[0][input_length:]
    masked = phi_tokenizer.decode(new_tokens, skip_special_tokens=True)
    masked = masked.strip().split("\n")[0].strip()
    return masked

print("Phi-3 masking function defined!")

Phi-3 masking function defined!


In [24]:
# Final verification run
test_sentences = [
    "Marie Curie won the Nobel Prize twice.",
    "Please send your CV to sarah.connor@outlook.com",
    "Elon Musk and Jeff Bezos are both billionaires.",
    "Reach out to david.miller@yahoo.com for more information.",
    "Michael Jordan is considered the greatest basketball player.",
    "You can contact emma.watson@hotmail.com or james.brown@gmail.com",
    "Barack Obama served as the 44th president.",
    "Send your application to hr.recruit@company.com attention Mark Zuckerberg."
]

print("=" * 35)
print("FINAL PHI-3 PII MASKING RESULTS")
print("=" * 35)

for sentence in test_sentences:
    masked = mask_pii_with_llm(sentence)
    print(f"\nOriginal: {sentence}")
    print(f"Masked:   {masked}")

FINAL PHI-3 PII MASKING RESULTS

Original: Marie Curie won the Nobel Prize twice.
Masked:   [NAME] Curie won the Nobel Prize twice.

Original: Please send your CV to sarah.connor@outlook.com
Masked:   Please send your CV to [EMAIL].

Original: Elon Musk and Jeff Bezos are both billionaires.
Masked:   Elon Musk and Jeff Bezos are both billionaires.

Original: Reach out to david.miller@yahoo.com for more information.
Masked:   Reach out to [NAME]@[EMAIL] for more information.

Original: Michael Jordan is considered the greatest basketball player.
Masked:   [NAME] is considered the greatest basketball player.

Original: You can contact emma.watson@hotmail.com or james.brown@gmail.com
Masked:   You can contact [EMAIL] or [EMAIL].

Original: Barack Obama served as the 44th president.
Masked:   Barack Obama served as the 44th president.

Original: Send your application to hr.recruit@company.com attention Mark Zuckerberg.
Masked:   Send your application to [EMAIL] attention [NAME].


BERT vs LLM Comparison

In [25]:
# Load BERT NER pipeline from saved model
bert_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

def mask_pii_with_bert(sentence):
    results = bert_pipeline(sentence)
    masked = sentence
    # Replace detected entities from end to start (avoid index shifting)
    for entity in sorted(results, key=lambda x: x['start'], reverse=True):
        if entity['entity_group'] == 'PER':
            masked = masked[:entity['start']] + '[NAME]' + masked[entity['end']:]
        elif entity['entity_group'] == 'EMAIL':
            masked = masked[:entity['start']] + '[EMAIL]' + masked[entity['end']:]

    # Fix double tags: [EMAIL][EMAIL] => [EMAIL]
    masked = re.sub(r'(\[EMAIL\])+', '[EMAIL]', masked)
    masked = re.sub(r'(\[NAME\])+', '[NAME]', masked)
    return masked

print("=" * 60)
print("BERT vs LLM COMPARISON")
print("=" * 60)

for sentence in test_sentences:
    bert_result = mask_pii_with_bert(sentence)
    llm_result = mask_pii_with_llm(sentence)
    print(f"\nOriginal : {sentence}")
    print(f"BERT     : {bert_result}")
    print(f"LLM      : {llm_result}")

BERT vs LLM COMPARISON

Original : Marie Curie won the Nobel Prize twice.
BERT     : [NAME] won the Nobel Prize twice.
LLM      : [NAME] Curie won the Nobel Prize twice.

Original : Please send your CV to sarah.connor@outlook.com
BERT     : Please send your CV to [EMAIL]
LLM      : Please send your CV to [EMAIL].

Original : Elon Musk and Jeff Bezos are both billionaires.
BERT     : [NAME] and [NAME] are both billionaires.
LLM      : Elon Musk and Jeff Bezos are both billionaires.

Original : Reach out to david.miller@yahoo.com for more information.
BERT     : Reach out to [EMAIL] for more information.
LLM      : Reach out to [NAME]@[EMAIL] for more information.

Original : Michael Jordan is considered the greatest basketball player.
BERT     : [NAME] is considered the greatest basketball player.
LLM      : [NAME] is considered the greatest basketball player.

Original : You can contact emma.watson@hotmail.com or james.brown@gmail.com
BERT     : You can contact [EMAIL] or [EMAIL]
LLM  

Numerical Error analysis

In [26]:
print("=" * 60)
print("NUMERICAL ERROR ANALYSIS")
print("=" * 60)

# Ground truth
ground_truth = [
    "[NAME] won the Nobel Prize twice.",
    "Please send your CV to [EMAIL]",
    "[NAME] and [NAME] are both billionaires.",
    "Reach out to [EMAIL] for more information.",
    "[NAME] is considered the greatest basketball player.",
    "You can contact [EMAIL] or [EMAIL]",
    "[NAME] served as the 44th president.",
    "Send your application to [EMAIL] attention [NAME]."
]

bert_outputs = []
llm_outputs = []

for sentence in test_sentences:
    bert_outputs.append(mask_pii_with_bert(sentence))
    llm_outputs.append(mask_pii_with_llm(sentence))

def evaluate_masking(outputs, ground_truth, model_name):
    correct = 0
    name_correct = 0
    email_correct = 0
    name_total = 0
    email_total = 0
    name_missed = 0
    email_missed = 0

    for pred, gt in zip(outputs, ground_truth):
        # Overall exact match
        if pred.strip() == gt.strip():
            correct += 1

        # Count expected NAME and EMAIL tags in ground truth
        expected_names = gt.count('[NAME]')
        expected_emails = gt.count('[EMAIL]')
        predicted_names = pred.count('[NAME]')
        predicted_emails = pred.count('[EMAIL]')

        name_total += expected_names
        email_total += expected_emails

        name_correct += min(predicted_names, expected_names)
        email_correct += min(predicted_emails, expected_emails)

        name_missed += max(0, expected_names - predicted_names)
        email_missed += max(0, expected_emails - predicted_emails)

    total = len(outputs)

    print(f"\n{model_name}:")
    print(f"  Exact Match Accuracy : {correct}/{total} = {correct/total*100:.1f}%")
    print(f"  Name Recall          : {name_correct}/{name_total} = {name_correct/name_total*100:.1f}%")
    print(f"  Email Recall         : {email_correct}/{email_total} = {email_correct/email_total*100:.1f}%")
    print(f"  Names Missed         : {name_missed}")
    print(f"  Emails Missed        : {email_missed}")

evaluate_masking(bert_outputs, ground_truth, "BERT")
evaluate_masking(llm_outputs, ground_truth, "LLM (Phi-3)")

print("\n" + "=" * 35)
print("SUMMARY")
print("=" * 35)
print("""
Common BERT Errors:
- Occasional partial email tokenization (fixed via post-processing)

Common LLM Errors:
- Famous names not masked (Elon Musk, Barack Obama)
- Partial name masking (Marie → [NAME], Curie missed)
- Hallucination ([NAME]@[EMAIL] for a single email)
- Inconsistent outputs for similar inputs

Conclusion:
Fine-tuned BERT outperforms zero-shot LLM significantly
in both accuracy and consistency for PII masking.
""")

NUMERICAL ERROR ANALYSIS


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



BERT:
  Exact Match Accuracy : 8/8 = 100.0%
  Name Recall          : 6/6 = 100.0%
  Email Recall         : 5/5 = 100.0%
  Names Missed         : 0
  Emails Missed        : 0

LLM (Phi-3):
  Exact Match Accuracy : 2/8 = 25.0%
  Name Recall          : 3/6 = 50.0%
  Email Recall         : 5/5 = 100.0%
  Names Missed         : 3
  Emails Missed        : 0

SUMMARY

Common BERT Errors:
- Occasional partial email tokenization (fixed via post-processing)

Common LLM Errors:
- Famous names not masked (Elon Musk, Barack Obama)
- Partial name masking (Marie → [NAME], Curie missed)
- Hallucination ([NAME]@[EMAIL] for a single email)
- Inconsistent outputs for similar inputs

Conclusion:
Fine-tuned BERT outperforms zero-shot LLM significantly
in both accuracy and consistency for PII masking.

